# Data Cleaning

## Business Objective

The objective of this notebook is to clean the dataset based on the findings from the Data Quality Assessment.

The cleaning process will focus on:

- Removing unnecessary duplicate records.
- Validating discount values.
- Preparing the dataset for exploratory data analysis (EDA).

At the end of this notebook, a cleaned dataset will be exported for further analysis.

## Import Required Libraries

The following libraries are imported to perform data cleaning operations.

In [1]:
import pandas as pd
import numpy as np

## Load Dataset

Load the original Myntra dataset into a Pandas DataFrame.

In [2]:
df = pd.read_csv("../data/myntra_dataset_ByScraping.csv")

## Verify Dataset

Display the first five rows to confirm that the dataset has been loaded correctly.


In [3]:
df.head()

,brand_name,pants_description,price,MRP,discount_percent,ratings,number_of_ratings
0,WROGN,Men Loose Fit Cotton Jeans,1374.0,2499.0,0.45,4.2,57.0
1,Flying Machine,Men Slim Fit Jeans,1829.0,2999.0,0.39,4.6,5.0
2,Roadster,Men Pure Cotton Jeans,974.0,2499.0,0.61,3.6,1100.0
3,Bene Kleed,Relaxed Fit Denim Jeans,873.0,2299.0,0.62,4.0,4800.0
4,Levis,Men 511 Slim Fit Jeans,1478.0,2899.0,0.49,4.3,264.0


### Observation

The dataset was loaded successfully.

The first five records confirm that all expected columns are available and the dataset is ready for the cleaning process.

# Cleaning Step 1 — Investigate Duplicate Rows

## Business Question

Are the duplicate rows exact duplicates or legitimate repeated product records?

Duplicate rows should never be removed without investigation because they may represent valid business records.

In [4]:
duplicate_rows = df[df.duplicated()]

## Display Duplicate Rows

Display a sample of duplicate records for manual inspection.

In [5]:
duplicate_rows.head(10)

,brand_name,pants_description,price,MRP,discount_percent,ratings,number_of_ratings
50,Mufti,Men Slim Fit Jeans,1649.0,3299.0,0.50,4.2,5.0
200,WROGN,Men Loose Fit Cotton Jeans,1374.0,2499.0,0.45,4.2,57.0
201,Flying Machine,Men Slim Fit Jeans,1829.0,2999.0,0.39,4.6,5.0
204,WROGN,Men Anti Fit Jeans,1623.0,2799.0,0.42,4.2,42.0
207,Roadster,Men Regular Fit Mid-Rise Jeans,759.0,1899.0,0.60,4.0,63.0
213,Levis,Men 511 Slim Fit Jeans,1478.0,2899.0,0.49,4.3,264.0
219,WROGN,Men Slim Fit Stretchable Jeans,1275.0,2199.0,0.42,4.2,130.0
225,Levis,Men 511 Slim Fit Jeans,2079.0,3999.0,0.48,3.9,31.0
231,WROGN,Men Anti Fit Jeans,1399.0,2799.0,0.50,4.1,25.0
234,Mufti,Men Mid-Rise Skinny Fit Jeans,1393.0,3399.0,0.59,3.8,16.0


### Observation

A total of **17,047 duplicate rows** were identified in the dataset.

Since duplicate records can inflate product counts and produce misleading analytical results, they require investigation before removal.

A manual inspection of sample duplicate records will help determine whether they are true duplicates or legitimate repeated product listings.

## Count Duplicate Rows

Determine the total number of duplicate records identified in the dataset.

In [6]:
duplicate_rows.shape

(17047, 7)

### Observation

The duplicate dataset contains **17,047 rows** and **7 columns**, confirming the number of duplicate records identified during the data quality assessment.

These records will be examined before any cleaning action is performed.

# Cleaning Step 2 — Remove Duplicate Rows

After confirming that the duplicate rows are exact duplicates, remove them from the dataset.

In [7]:
df = df.drop_duplicates()

## Verify Dataset Size

Check the dataset dimensions after removing duplicate records.

In [8]:
df.shape

(35073, 7)

### Observation

Duplicate rows were successfully removed from the dataset.

The dataset now contains **35,073 unique product records**, ensuring that subsequent analyses are not affected by duplicate entries.

Removing duplicate records improves the accuracy and reliability of business insights.

# Cleaning Step 3 — Investigate Discount Values

## Business Question

Are discount values stored as decimal values (0.40) or percentages (40)?

Understanding the storage format is important before performing any cleaning or calculations.

## Summary Statistics for Discount

Review the statistical summary of the discount column.

In [9]:
df["discount_percent"].describe()

count    35073.000000
mean         2.039860
std          5.446518
min          0.020000
25%          0.400000
50%          0.550000
75%          0.650000
max         64.000000
Name: discount_percent, dtype: float64

### Observation

The summary statistics provide an overview of the minimum, maximum, average, and distribution of discount values.

These statistics help determine whether the discount values follow the expected format or require additional investigation.

## Display Sample Discount Values

Display the first unique discount values to understand how discounts are represented in the dataset.

In [10]:
df["discount_percent"].unique()[:20]

array([0.45, 0.39, 0.61, 0.62, 0.49, 0.53, 0.55, 0.42, 0.7 , 0.6 , 0.67,
       0.75, 0.63, 0.72, 0.34, 0.48, 0.74, 0.38, 0.64, 0.5 ])

### Observation

Most discount values are stored as decimals between 0 and 1 (e.g., 0.45 for 45% off).

However, some records contain values greater than 1 (up to 64.0), indicating an inconsistent scraper format that cannot be reliably converted to decimals.

These inconsistent records will be removed in the next step.

## Display Highest Discount Values

Inspect the largest discount values available in the dataset.

In [11]:
df["discount_percent"].sort_values().tail(20)

28831    50.05
40260    50.05
33112    50.83
33444    50.83
33113    50.83
124      50.88
697      50.88
16908    51.00
16894    51.00
12546    52.00
13347    52.00
44429    52.53
44430    52.53
14465    52.59
14527    54.16
12026    58.25
11279    58.76
11042    58.83
11452    58.98
29178    64.00
Name: discount_percent, dtype: float64

### Observation

The highest discount values were reviewed to identify any unusual or potentially invalid entries.

Further validation will determine whether these values require cleaning or represent valid business data.

## Remove Inconsistent Discount Values

Some products contain discount values **greater than 1**, outside the expected decimal range (0–1). These values (1.1–64.0) stem from inconsistent scraper output and cannot be reliably converted — dividing by 100 does not match price-derived discounts.

Records with `discount_percent > 1` are removed to ensure accurate analysis.

This step improves data quality and prevents misleading business insights.

In [12]:
# Remove inconsistent discount values (scraper format outside 0-1 decimal range)

df = df[
    (df["discount_percent"] >= 0) &
    (df["discount_percent"] <= 1)
].copy()

print(df.shape)

(31527, 7)


### Observation

**3,546 records** with `discount_percent > 1` were removed. These values (1.1–64.0) reflect an inconsistent scraper format that cannot be reliably converted — dividing by 100 does not match price-derived discounts.

The dataset now contains **31,527 clean records** ready for analysis.

# Cleaning Step 4 — Export Clean Dataset

Save the cleaned dataset for use in the Exploratory Data Analysis notebook.

In [13]:
df.to_csv("../data/myntra_cleaned.csv", index=False)

print("Clean dataset exported successfully!")

Clean dataset exported successfully!


### Observation

The cleaned dataset has been exported successfully as **myntra_cleaned.csv**.

This dataset will be used for all subsequent analysis to ensure consistency throughout the project.